# Prompt Injection Guard v0.1
Reproducible Kaggle training, MLflow tracking, release-gate evaluation, and Hugging Face publishing.

## 1. Environment
Enable Internet in Kaggle. Add `HF_TOKEN` as a Kaggle secret only when publishing a passing release.

In [ ]:
!pip install -q -r /kaggle/input/ai-security-gateway-source/training/requirements-kaggle.txt

## 2. Configuration and secrets

In [ ]:
import os, sys
from kaggle_secrets import UserSecretsClient
SOURCE = '/kaggle/input/ai-security-gateway-source'
sys.path.insert(0, SOURCE)
os.environ['MLFLOW_DIR'] = '/kaggle/working/mlruns'
os.environ['HF_MODEL_REPO'] = 'YOUR_HF_USERNAME/prompt-injection-guard-v1'
try:
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    print('HF_TOKEN not configured; training will still run without upload.')

## 3. Generate the synthetic PII validation corpus
These records test deterministic recognizers; they do not train a PII model.

In [ ]:
from pathlib import Path
from training.generate_pii_dataset import generate
from training.evaluate_pii import write_evaluation
pii_rows = generate(size=3000, seed=42)
pii_evaluation = write_evaluation(
    pii_rows,
    Path(SOURCE) / 'config' / 'pii_rules.yaml',
    Path('/kaggle/working/pii_evaluation.json'),
)
pii_evaluation

## 4. Train, evaluate, and log with MLflow
The training function loads Deepset, adds difficult benign examples, deduplicates, calibrates the threshold on validation data, evaluates untouched test sets, and verifies the exported artifact.

In [ ]:
from dataclasses import asdict
from training.train import run_training
result = run_training('/kaggle/working/release', upload=True)
asdict(result)

## 5. Inspect the release
If the release gate fails, the files remain available as experimental notebook output but are not uploaded.

In [ ]:
import json
from pathlib import Path
release = Path('/kaggle/working/release')
print((release / 'evaluation.json').read_text())
print('Files:', [path.name for path in release.iterdir()])
print('HF commit:', result.hf_commit)

## 6. Preserve MLflow output

In [ ]:
!cd /kaggle/working && zip -qr mlruns.zip mlruns && ls -lh mlruns.zip release/